In [3]:
import os
import json
from redis import Redis

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    message_to_dict,
    messages_from_dict
)

from dotenv import load_dotenv
load_dotenv()

TypeError: ForwardRef.__init__() got an unexpected keyword argument 'is_class'

In [2]:
%pip install redis python-dotenv langchain-core==0.3.86 langchain-openai==0.3.35 httpx

  Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
  Using cached packaging-25.0-py3-none-any.whl (66 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)
  Using cached h11-0.16.0-py3-none-any.whl (37 kB)
  Using cached distro-1.9.0-py3-none-any.whl (20 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
  Attempting uninstall: packaging
    Found existing installation: packaging 26.3
    Uninstalling packaging-26.3:
      Successfully uninstalled packaging-26.3
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [4]:
os.environ["OPEN_ROUTER_API_KEY"] = os.getenv("OPEN_ROUTER_API_KEY")
os.environ["REDIS_URL"] = os.getenv("REDIS_URL")

In [5]:




redis_client = Redis.from_url(
    os.environ["REDIS_URL"],
    decode_responses=True
)

redis_client.ping()

True

In [6]:
llm = ChatOpenAI(
    model="google/gemini-2.5-flash",
    api_key=os.environ["OPEN_ROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    timeout=60,
    max_retries=0,
)

In [7]:
human_message = HumanMessage(
    content="My name is Sameer."
)

ai_message = AIMessage(
    content="Hello Sameer."
)

In [8]:
human_json= json.dumps(message_to_dict(human_message))

In [9]:
ai_json = json.dumps(message_to_dict(ai_message))

In [10]:




redis_client = Redis.from_url(
    os.environ["REDIS_URL"],
    decode_responses=True
)

redis_client.ping()

True

In [18]:
redis_client.lpush("user_id:session_id", human_json)
redis_client.lpush("user_id:session_id", ai_json)

4

In [1]:
## Getting the messages from Redis

In [11]:
redis_client

Redis<ConnectionPool<SSLConnection<host=pet-blowfish-191113.upstash.io,port=6379,db=0>>>

In [12]:
MESSAGES_KEY="user_id:session_id"
stored_json = redis_client.lrange(
    MESSAGES_KEY,
    0,
    -1
)
stored_json

['{"type": "ai", "data": {"content": "Hello Sameer.", "additional_kwargs": {}, "response_metadata": {}, "type": "ai", "name": null, "id": null, "example": false, "tool_calls": [], "invalid_tool_calls": [], "usage_metadata": null}}',
 '{"type": "human", "data": {"content": "My name is Sameer.", "additional_kwargs": {}, "response_metadata": {}, "type": "human", "name": null, "id": null, "example": false}}',
 '{"type": "ai", "data": {"content": "Hello Sameer.", "additional_kwargs": {}, "response_metadata": {}, "type": "ai", "name": null, "id": null, "tool_calls": [], "invalid_tool_calls": [], "usage_metadata": null}}',
 '{"type": "human", "data": {"content": "My name is Sameer.", "additional_kwargs": {}, "response_metadata": {}, "type": "human", "name": null, "id": null}}']

In [13]:
stored_dicts = [
    json.loads(item)
    for item in stored_json
]
stored_dicts

[{'type': 'ai',
  'data': {'content': 'Hello Sameer.',
   'additional_kwargs': {},
   'response_metadata': {},
   'type': 'ai',
   'name': None,
   'id': None,
   'example': False,
   'tool_calls': [],
   'invalid_tool_calls': [],
   'usage_metadata': None}},
 {'type': 'human',
  'data': {'content': 'My name is Sameer.',
   'additional_kwargs': {},
   'response_metadata': {},
   'type': 'human',
   'name': None,
   'id': None,
   'example': False}},
 {'type': 'ai',
  'data': {'content': 'Hello Sameer.',
   'additional_kwargs': {},
   'response_metadata': {},
   'type': 'ai',
   'name': None,
   'id': None,
   'tool_calls': [],
   'invalid_tool_calls': [],
   'usage_metadata': None}},
 {'type': 'human',
  'data': {'content': 'My name is Sameer.',
   'additional_kwargs': {},
   'response_metadata': {},
   'type': 'human',
   'name': None,
   'id': None}}]

In [14]:
messages_from_dict = messages_from_dict(stored_dicts)
messages_from_dict

[AIMessage(content='Hello Sameer.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='My name is Sameer.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello Sameer.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='My name is Sameer.', additional_kwargs={}, response_metadata={})]